# **Epigenetics: *GPNMB*+ vs Homeostatic cells**

In this notebook, we compare the epigenetic profiles of GPNMB+ and homeostatic microglial nuclei across Datasets 8 and 9.

In [ ]:
# Import libraries

# ==============================================================================
# STANDARD AND DATA SCIENCE LIBRARIES
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle  

# ==============================================================================
# STATISTICAL AND MATHEMATICAL MODULES
# ==============================================================================
from scipy.stats import combine_pvalues, ttest_rel
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

In [ ]:
# Read adata files (Datasets 8 & 9, paired RNA- and ATAC-seq)

ds8 = sc.read_h5ad('Dataset_8.h5ad')
ds9 = sc.read_h5ad('Dataset_9.h5ad')
ds8_atac = sc.read_h5ad('Dataset_8_ATAC.h5ad')
ds9_atac = sc.read_h5ad('Dataset_9_ATAC.h5ad')

In [ ]:
# Read the LINGER output file (for Dataset 8 only)
df_linger_data = pd.read_csv("LINGER_cis.txt", sep="\t")

# Read the reference GTF annotation file directly from the dataset folder
df_gtf = pd.read_csv(
    "genomic.gtf", 
    sep="\t", 
    comment="#", 
    names=["chrom", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"]
)

# Extract gene_name from the 'attribute' column for clean mapping inside the function
# This ensures df_gtf['gene_name'] column exists and filters properly
df_gtf["gene_name"] = df_gtf["attribute"].str.extract(r'gene_id "([^"]+)"')

In [ ]:
ds8

In [ ]:
ds8_atac

In [ ]:
ds9

In [ ]:
ds9_atac

Dataset 8 contains 6,912 nuclei, and Dataset 9 contains 18,782 nuclei, both profiled using paired single-nucleus RNA-seq and ATAC-seq.

In [ ]:
def plot_custom_dotplot(adata, gene_list, groupby='leiden_res_2.0', title="Custom Gene Set"):
    """
    Generates a standardized dotplot for a custom list of genes and forces gene names to be italicized.
    """
    # 1. Prepare data object for plotting by deduplicating gene symbols
    adata_plot = adata.copy()
    adata_plot.var_names = adata_plot.var_names.astype(str)
    adata_plot = adata_plot[:, ~adata_plot.var_names.duplicated(keep='first')].copy()

    # 2. Filter the list to include only genes present in the dataset matrix
    genes = [g for g in gene_list if g in adata_plot.var_names]

    if not genes:
        print(f"⚠️ Warning: None of the target genes were found in the dataset.")
        return

    # 3. Call the Scanpy dotplot engine. show=False is mandatory for downstream axis modification.
    dp = sc.pl.dotplot(
        adata_plot,
        var_names=genes,
        groupby=groupby,
        use_raw=False,
        standard_scale='var',
        title=title,
        figsize=(12, 4),
        dendrogram=False,
        cmap='viridis',
        show=False
    )

    # 4. Safe check for axes dictionary structure across different Scanpy versions
    if isinstance(dp, dict):
        ax = dp['mainplot_ax']
    else:
        ax = dp.get_axes()['mainplot_ax']
        
    # 5. Force x-axis tick labels (gene symbols) to be italicized for publication standard
    labels = [label.get_text() for label in ax.get_xticklabels()]
    ax.set_xticklabels(labels, fontdict={'style': 'italic'})

    plt.show()

# Target gene panel for myeloid lineage profiling
target_genes = [
    # Macrophage markers
    "F13A1", "CD163", "CD163L1", "COLEC12",

    # Uncategorized transition genes
    "GRID2", "CCDC26",

    # Homeostatic
    "P2RY12", "CX3CR1", "SORL1", "MEF2A", "ITPR2", "FRMD4A", "ELMO1", "ANKRD44", "SRGAP2",

    # GPNMB+ signature
    "GPNMB", "MITF", "PPARG", "PTPRG", "MYO1E", "CPM", "KCNMA1", "ATG7", "IQGAP2",
    "STARD13", "DPYD", "LRRK2", "FOXP1", "APOE", "SERPINE1",

    # Inflammation & Related
    "SPP1", "TMEM163", "MSR1", "SLC11A1", "CD83", "IL1B", "IRAK2",

    # Ribosomal & Immune
    "RPL32", "RPS19", "C1QB", "FTH1", "TMSB10", "HLA-B",

    # HSPs
    "HSPH1", "DNAJB1", "HSP90AA1"
]

In [ ]:
plot_custom_dotplot(ds8, target_genes, title="Dataset 8: Myeloid Lineage & Activation Markers")

In Dataset 8, Clusters 0 and 1 exhibit the highest expression of homeostatic genes (e.g., ITPR2, FRMD4A), whereas Cluster 2 displays the strongest enrichment for the GPNMB⁺ signature. Cluster 5 is characterized by a macrophage phenotype (F13A1⁺).

In [ ]:
plot_custom_dotplot(ds9, target_genes, title="Dataset 9: Myeloid Lineage & Activation Markers")

In Dataset 9, Clusters 2, 4, and 5 exhibit the highest expression of homeostatic genes (ranging from P2RY12 to SRGAP2). Cluster 11 displays the strongest enrichment for GPNMB and associated genes; however, due to prominent F13A1⁺ expression, these cells represent macrophages. Consequently, Clusters 6 and 8 are annotated as GPNMB⁺ microglia with moderate expression of signature activation markers.

# **Comparing the number of peaks per cell in GPNMB+ and homeostatic microglia**

First, we evaluated the distribution of the number of peaks per cell within each pre-selected cluster.

In [ ]:
# ==============================================================================
# 1. COMPUTE SINGLE-CELL DATA AND DONOR-LEVEL MEANS
# ==============================================================================
# --- Dataset 8 Processing ---
ds8_atac.obs["n_peaks"] = np.asarray(ds8_atac.X.sum(axis=1)).flatten()
d8_filtered_obs = ds8_atac.obs[ds8_atac.obs["leiden_res_2.0"].isin(["0", "1", "2"])].copy()

df8_cells = pd.DataFrame({
    "n_peaks": d8_filtered_obs["n_peaks"],
    "cluster": d8_filtered_obs["leiden_res_2.0"].apply(lambda x: f"D8C{x}"),
    "color_group": d8_filtered_obs["leiden_res_2.0"].apply(lambda x: "red" if x == "2" else "blue")
})
# Aggregate to find the true mean per cluster for each sample/donor cohort
df8_donors = d8_filtered_obs.groupby(["sample", "leiden_res_2.0"])["n_peaks"].mean().reset_index()
df8_donors["cluster"] = df8_donors["leiden_res_2.0"].apply(lambda x: f"D8C{x}")
df8_donors["color_group"] = df8_donors["leiden_res_2.0"].apply(lambda x: "red" if x == "2" else "blue")

# --- Dataset 9 Processing ---
ds9_atac.obs["n_peaks"] = np.asarray(ds9_atac.X.sum(axis=1)).flatten()
d9_filtered_obs = ds9_atac.obs[ds9_atac.obs["leiden_res_2.0"].isin(["2", "4", "5", "6", "8"])].copy()

df9_cells = pd.DataFrame({
    "n_peaks": d9_filtered_obs["n_peaks"],
    "cluster": d9_filtered_obs["leiden_res_2.0"].apply(lambda x: f"D9C{x}"),
    "color_group": d9_filtered_obs["leiden_res_2.0"].apply(lambda x: "red" if x in ["6", "8"] else "blue")
})
# Aggregate to find the true mean per cluster for each unique donor
df9_donors = d9_filtered_obs.groupby(["sample", "leiden_res_2.0"])["n_peaks"].mean().reset_index()
df9_donors["cluster"] = df9_donors["leiden_res_2.0"].apply(lambda x: f"D9C{x}")
df9_donors["color_group"] = df9_donors["leiden_res_2.0"].apply(lambda x: "red" if x in ["6", "8"] else "blue")

# ==============================================================================
# 2. PLOT COMBINED CELL-VIOLIN + DONOR-DOTS FIGURE
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=False, dpi=300)

_C_GPNMB = "#ff9287"  # Crimson / Red
_C_HOMEO = "#8bd1ff"  # Cerulean / Blue
palette = {"red": _C_GPNMB, "blue": _C_HOMEO}

# --- Panel 1: Dataset 8 (Cells + Sample Cohorts) ---
order_d8 = ["D8C0", "D8C1", "D8C2"]
sns.violinplot(
    data=df8_cells, x="cluster", y="n_peaks", order=order_d8, hue="color_group",
    palette=palette, dodge=False, inner="quartile", linewidth=1.5, ax=ax1
)
# Overlay sample cohort averages as prominent, semi-transparent black dots
sns.stripplot(
    data=df8_donors, x="cluster", y="n_peaks", order=order_d8,
    color="black", alpha=0.8, size=6, jitter=0.15, ax=ax1
)
ax1.set_ylim(-500, 300000)
ax1.set_title("Dataset 8", fontsize=16, fontweight="bold")
ax1.set_xlabel("")
ax1.set_ylabel("Number of Accessible Peaks", fontsize=14)
ax1.tick_params(labelsize=12)
if ax1.get_legend(): ax1.get_legend().remove()

# --- Panel 2: Dataset 9 (Cells + Individual Donors) ---
order_d9 = ["D9C2", "D9C4", "D9C5", "D9C6", "D9C8"]
sns.violinplot(
    data=df9_cells, x="cluster", y="n_peaks", order=order_d9, hue="color_group",
    palette=palette, dodge=False, inner="quartile", linewidth=1.5, ax=ax2
)
# Overlay true individual donor averages as prominent, semi-transparent black dots
sns.stripplot(
    data=df9_donors, x="cluster", y="n_peaks", order=order_d9,
    color="black", alpha=0.8, size=5, jitter=0.2, ax=ax2
)
ax2.set_ylim(-5000, 300000)
ax2.set_title("Dataset 9", fontsize=16, fontweight="bold")
ax2.set_xlabel("")
ax2.set_ylabel("")
ax2.tick_params(labelsize=12)
if ax2.get_legend(): ax2.get_legend().remove()

# Configure comprehensive multi-panel layout
fig.suptitle("Chromatin Accessibility (n_peaks) Profiles with Donor-level Overlays", 
             fontsize=18, fontweight="bold", y=1.02)

sns.despine(fig=fig)
plt.tight_layout()
plt.show()

In both datasets, GPNMB+ cells appear to have a higher average number of peaks compared to homeostatic cells. To verify the statistical significance of this difference and avoid pseudoreplication, we applied a linear mixed-effects model (LMM) that accounts for donor information. The model was fitted independently for each dataset.

In [ ]:
# ==============================================================================
# 1. LINEAR MIXED-EFFECTS MODEL (LMM) FOR DATASET 8
# ==============================================================================
print("Processing Linear Mixed-Effects Model for Dataset 8...")

# Compute peak sums directly from the matrix and filter metadata
n_peaks_vector_8 = np.asarray(ds8_atac.X.sum(axis=1)).flatten()
peaks_dict_8 = dict(zip(ds8_atac.obs_names, n_peaks_vector_8))
d8_filtered = ds8_atac.obs[ds8_atac.obs['leiden_res_2.0'].isin(['0', '1', '2'])].copy()

# Construct regression modeling dataframe
df_lmm_8 = pd.DataFrame({
    'n_peaks': d8_filtered.index.map(peaks_dict_8).astype(float),
    'is_activated': d8_filtered['leiden_res_2.0'].apply(lambda x: 1 if x == '2' else 0),
    'sample_id': d8_filtered['sample'].astype(str)
})

# Initialize and fit LMM with sample nested random effects
model_8 = smf.mixedlm("n_peaks ~ is_activated", data=df_lmm_8, groups=df_lmm_8["sample_id"])
mdf_8 = model_8.fit()

# Extract statistical parameters for reporting
p_value_8 = mdf_8.pvalues['is_activated']
coef_8 = mdf_8.params['is_activated']
stderr_8 = mdf_8.bse['is_activated']

# ==============================================================================
# 2. LINEAR MIXED-EFFECTS MODEL (LMM) FOR DATASET 9
# ==============================================================================
print("Processing Linear Mixed-Effects Model for Dataset 9...")

# Compute peak sums directly from the matrix and filter metadata
n_peaks_vector_9 = np.asarray(ds9_atac.X.sum(axis=1)).flatten()
peaks_dict_9 = dict(zip(ds9_atac.obs_names, n_peaks_vector_9))
d9_filtered = ds9_atac.obs[ds9_atac.obs['leiden_res_2.0'].isin(['2', '4', '5', '6', '8'])].copy()

# Construct regression modeling dataframe
df_lmm_9 = pd.DataFrame({
    'n_peaks': d9_filtered.index.map(peaks_dict_9).astype(float),
    'is_activated': d9_filtered['leiden_res_2.0'].apply(lambda x: 1 if x in ['6', '8'] else 0),
    'sample_id': d9_filtered['sample'].astype(str)
})

# Initialize and fit LMM with sample nested random effects
model_9 = smf.mixedlm("n_peaks ~ is_activated", data=df_lmm_9, groups=df_lmm_9["sample_id"])
mdf_9 = model_9.fit()

# Extract statistical parameters for reporting
p_value_9 = mdf_9.pvalues['is_activated']
coef_9 = mdf_9.params['is_activated']
stderr_9 = mdf_9.bse['is_activated']

# ==============================================================================
# 3. META-ANALYSIS P-VALUE COMBINATION
# ==============================================================================
print("Performing Meta-Analysis across cohorts...")

# Empirical p-values obtained directly from the fitted LMMs above
empirical_p_values = [p_value_8, p_value_9]

# Method A: Unweighted Fisher's Meta-Analysis
_, p_fisher = combine_pvalues(empirical_p_values, method='fisher')

# Method B: Sample size-weighted Stouffer's Meta-Analysis
cell_weights = [len(df_lmm_8), len(df_lmm_9)]
_, p_stouffer = combine_pvalues(empirical_p_values, method='stouffer', weights=cell_weights)

# ==============================================================================
# 4. CONSOLIDATED MANUSCRIPT STATISTICAL REPORT
# ==============================================================================
print("\n" + "="*70)
print("             MANUSCRIPT META-ANALYSIS SUMMARY REPORT")
print("="*70)
print(f"Dataset 8 LMM Fixed Effect (Coef): {coef_8:.4f} (SE: {stderr_8:.4f})")
print(f"Dataset 8 Empirical P-value:      {p_value_8:.4e}")
print(f"Dataset 8 Analytical Cell Count:   {cell_weights[0]}")
print("-"*70)
print(f"Dataset 9 LMM Fixed Effect (Coef): {coef_9:.4f} (SE: {stderr_9:.4f})")
print(f"Dataset 9 Empirical P-value:      {p_value_9:.4e}")
print(f"Dataset 9 Analytical Cell Count:   {cell_weights[1]}")
print("="*70)
print(f"Combined P-value (Fisher's Unweighted):   {p_fisher:.4e}")
print(f"Combined P-value (Weighted Stouffer's):   {p_stouffer:.4e}")
print("="*70)

The LMM confirmed that the difference is statistically significant in both datasets (p-value = 1.1796e-03 and 6.3015e-03 for Datasets 8 and 9, respectively). The combined p-value is 9.5214e-05 according to Fisher's method, and 3.3444e-04 according to weighted Stouffer's. Thus, we conclude that chromatin in GPNMB+ microglia is generally more open, even when taking donor information into account, and this is a highly significant result.

# **Finding differentially available regions**

Second, we identified marker differentially accessible regions (DARs) that define either the GPNMB+ or homeostatic populations. We used the Wilcoxon rank-sum test, irrespective of donors, which is the same method commonly applied to find cluster marker genes in snRNA-seq.

Our baseline assumption was that signature genes for GPNMB+ microglia (such as GPNMB, IQGAP2, and DPYD) would exhibit higher accessibility, whereas homeostatic loci (such as P2RY12) would be more open in homeostatic nuclei.

In [ ]:
# ==============================================================================
# DIFFERENTIAL ACCESSIBILITY REGION (DAR) ANALYSIS
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. DATASET 8: Cluster 2 (Activated) vs Clusters 0 + 1 (Homeostasis)
# ------------------------------------------------------------------------------
print("Running DAR analysis for Dataset 8...")

# Create a logical mask to isolate only target clusters
cells_subset_d8 = ds8_atac.obs["leiden_res_2.0"].isin(["0", "1", "2"])
adata_subset_d8 = ds8_atac[cells_subset_d8].copy()

# Collapse Clusters 0 and 1 into a single reference group named 'Homeostasis'
adata_subset_d8.obs["comparison_group"] = adata_subset_d8.obs["leiden_res_2.0"].map({
    "0": "Homeostasis",
    "1": "Homeostasis",
    "2": "Activated"
})

# Execute Differential Accessibility analysis using Wilcoxon rank-sum test
sc.tl.rank_genes_groups(
    adata_subset_d8,
    groupby="comparison_group",
    groups=["Activated"],
    reference="Homeostasis",
    method="wilcoxon",
    use_raw=False
)

# Extract statistics into a clean DataFrame
result_d8 = sc.get.rank_genes_groups_df(adata_subset_d8, group="Activated")
print(f"✅ Dataset 8: Found {len(result_d8[result_d8['pvals_adj'] < 0.05])} significant peaks (FDR < 0.05)")


# ------------------------------------------------------------------------------
# 2. DATASET 9: Clusters 6 + 8 (Activated) vs Clusters 2 + 4 + 5 (Homeostasis)
# ------------------------------------------------------------------------------
print("\nRunning DAR analysis for Dataset 9...")

# Create a logical mask to isolate only target clusters
target_clusters_d9 = ["2", "4", "5", "6", "8"]
cells_subset_d9 = ds9_atac.obs["leiden_res_2.0"].isin(target_clusters_d9)
adata_subset_d9 = ds9_atac[cells_subset_d9].copy()

# Map specific clusters to 'Activated' and 'Homeostasis' groups respectively
adata_subset_d9.obs["comparison_group"] = adata_subset_d9.obs["leiden_res_2.0"].map({
    "2": "Homeostasis",
    "4": "Homeostasis",
    "5": "Homeostasis",
    "6": "Activated",
    "8": "Activated"
})

# Execute Differential Accessibility analysis using Wilcoxon rank-sum test
sc.tl.rank_genes_groups(
    adata_subset_d9,
    groupby="comparison_group",
    groups=["Activated"],
    reference="Homeostasis",
    method="wilcoxon",
    use_raw=False
)

# Extract statistics into a clean DataFrame
result_d9 = sc.get.rank_genes_groups_df(adata_subset_d9, group="Activated")
print(f"✅ Dataset 9: Found {len(result_d9[result_d9['pvals_adj'] < 0.05])} significant peaks (FDR < 0.05)")

# Map peak coordinates to gene annotations from .var dataframe
result_d8["gene_annotation"] = result_d8["names"].map(ds8_atac.var["gene_ann"])
result_d9["gene_annotation"] = result_d9["names"].map(ds9_atac.var["gene_ann"])

# Save the dataframes
result_d8.to_csv("Dataset_8_DARs_Results.csv", index=False)
result_d9.to_csv("Dataset_9_DARs_Results.csv", index=False)

In [ ]:
# Plot volcano plots

def plot_single_volcano(df, ax, title, selection_criterion, p_threshold=0.05, lfc_threshold=0.5):
    """
    Generates a stylized volcano plot and appends top differentially accessible 
    elements into dedicated clean text tables inside the plot area.
    """
    df = df.copy()
    # Calculate negative log10 p-value (cap at 300 to prevent infinity issues)
    df['log_p'] = -np.log10(df['pvals_adj'] + 1e-300)
    
    # Define groups for coloring
    df['significance'] = 'Non-significant'
    df.loc[(df['pvals_adj'] < p_threshold) & (df['logfoldchanges'] > lfc_threshold), 'significance'] = 'Upregulated'
    df.loc[(df['pvals_adj'] < p_threshold) & (df['logfoldchanges'] < -lfc_threshold), 'significance'] = 'Downregulated'
    
    # Background scatter plot
    color_map = {'Non-significant': '#bdc3c7', 'Upregulated': '#e74c3c', 'Downregulated': '#3498db'}
    sns.scatterplot(
        data=df, x='logfoldchanges', y='log_p', hue='significance', 
        palette=color_map, alpha=0.5, s=12, edgecolor=None, ax=ax
    )
    
    # Reference threshold lines
    ax.axhline(-np.log10(p_threshold), color='black', linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(lfc_threshold, color='black', linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(-lfc_threshold, color='black', linestyle='--', linewidth=1, alpha=0.7)
    
    # Filter out Intergenic and NaN for clean genetic labeling
    labeled_pool = df[(df['gene_annotation'].notna()) & (df['gene_annotation'] != 'Intergenic')]
    
    # Select candidate genes based on the specified criterion
    if selection_criterion == 'p_value':
        top_up = labeled_pool[labeled_pool['significance'] == 'Upregulated'].nlargest(10, 'log_p')
        top_down = labeled_pool[labeled_pool['significance'] == 'Downregulated'].nlargest(10, 'log_p')
        sub_title = f"{title}\n[Top 10 labeled by Significance (p-value)]"
    elif selection_criterion == 'log_fc':
        top_up = labeled_pool[labeled_pool['significance'] == 'Upregulated'].nlargest(10, 'logfoldchanges')
        top_down = labeled_pool[labeled_pool['significance'] == 'Downregulated'].nsmallest(10, 'logfoldchanges')
        sub_title = f"{title}\n[Top 10 labeled by Magnitude (Log2FC)]"
        
    # Generate clean text blocks to display as side tables within the plot layout
    up_genes_list = "\n".join([f"• {g}" for g in top_up['gene_annotation'].unique() if pd.notna(g)])
    down_genes_list = "\n".join([f"• {g}" for g in top_down['gene_annotation'].unique() if pd.notna(g)])
    
    # Common bounding box styling for the tabular inserts
    props = dict(boxstyle='round,pad=0.5', facecolor='#ffffff', edgecolor='#dcdde1', alpha=0.95, lw=1)
    
    # Render Upregulated genes table in the upper-right corner area
    if up_genes_list:
        ax.text(
            0.95, 0.92, f"Top Upregulated:\n{up_genes_list}", 
            transform=ax.transAxes, fontsize=8.5, color='#c0392b', fontweight='bold',
            verticalalignment='top', horizontalalignment='right', bbox=props
        )
        
    # Render Downregulated genes table in the upper-left corner area
    if down_genes_list:
        ax.text(
            0.05, 0.92, f"Top Downregulated:\n{down_genes_list}", 
            transform=ax.transAxes, fontsize=8.5, color='#2980b9', fontweight='bold',
            verticalalignment='top', horizontalalignment='left', bbox=props
        )
        
    # Panel layout polish
    ax.set_title(sub_title, fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Log2 Fold Change', fontsize=12)
    ax.set_ylabel('-Log10 Adjusted P-value', fontsize=12)
    ax.tick_params(labelsize=11)
    if ax.get_legend(): ax.get_legend().remove()

# ==============================================================================
# INITIALIZE THE 2x2 FIGURATION GRID FOR MANUSCRIPT DATA
# ==============================================================================
fig, axs = plt.subplots(2, 2, figsize=(18, 14), dpi=300)

# Row 1: Dataset 8 (By P-value on the left, By Log2FC on the right)
plot_single_volcano(result_d8, axs[0, 0], title="Dataset 8", selection_criterion='p_value')
plot_single_volcano(result_d8, axs[0, 1], title="Dataset 8", selection_criterion='log_fc')

# Row 2: Dataset 9 (By P-value on the left, By Log2FC on the right)
plot_single_volcano(result_d9, axs[1, 0], title="Dataset 9", selection_criterion='p_value')
plot_single_volcano(result_d9, axs[1, 1], title="Dataset 9", selection_criterion='log_fc')

# Add a unified layout legend at the very top
handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=3, fontsize=12, frameon=False)

sns.despine(fig=fig)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

As inferred from the preliminary analysis, GPNMB+ microglia indeed show higher accessibility at GPNMB+ signature genes (though not within the top 10). Additionally, S100 genes are among the top 10 GPNMB+-enriched loci in Dataset 8, which is interesting given their established role in aging and inflammation. Conversely, in homeostatic microglia, we found almost no DARs, with the notable exception of MEIS1, an important repressor according to our TF-target analysis. This aligns with our previous finding that the homeostatic state is generally characterized by closed chromatin.

We examined each target genomic region independently.

In [ ]:
# Define a function to illustrate a region with features

def plot_region_combined_with_donors(
    df_gtf,
    ds8_atac,
    ds9_atac,
    result8,
    result9,
    chrom,
    start,
    end,
    df_linger=None,
    genes=None,
    fdr_threshold=0.05,
    figsize=(16, 12),
):
    offset, width = start, end - start

    # --- STYLE CONFIGURATION & PALETTE ---
    C_RED_EXON, C_RED_ARROW = "#b294df", "#7733c2"
    C_BLU_EXON, C_BLU_ARROW = "#b294df", "#7733c2"
    _C_GPNMB, _C_HOMEO, _C_NS = "#c0392b", "#2980b9", "#aaaaaa"
    FS_TITLE, FS_LABELS, FS_GENES, FS_SMALL = 23, 23, 18, 18

    # 1. Genomic Features (GTF Processing)
    region_gtf = df_gtf[
        (df_gtf["chrom"] == chrom)
        & (df_gtf["start"] <= end)
        & (df_gtf["end"] >= start)
    ].copy()
    if genes:
        region_gtf = region_gtf[region_gtf["gene_name"].isin(genes)]

    gene_list = sorted(
        region_gtf[region_gtf["feature"] == "gene"]["gene_name"].unique(),
        key=lambda g: region_gtf[
            (region_gtf["gene_name"] == g) & (region_gtf["feature"] == "gene")
        ]["start"].min(),
    )

    # 2. Chromatin Signal Extraction
    var_names = ds8_atac.var_names.to_series()
    p_coords = var_names.str.extract(r"^(.+):(\d+)-(\d+)$").rename(
        columns={0: "chr", 1: "s", 2: "e"}
    )
    p_coords[["s", "e"]] = p_coords[["s", "e"]].astype(int)

    region_peaks = var_names[
        (p_coords["chr"] == chrom)
        & (p_coords["s"] <= end)
        & (p_coords["e"] >= start)
    ].tolist()

    # Dynamic parsing of LINGER cis-regulatory elements from file database
    linger_cis_peaks = set()
    if df_linger is not None:
        # Assuming df_linger has 'Peak' column or coordinates matching the region
        # Adjust column name mapping if your LINGER file uses a different header (e.g., 'peak_id')
        linger_col = "Peak" if "Peak" in df_linger.columns else df_linger.columns[0]
        linger_cis_peaks = set(
            df_linger[df_linger[linger_col].isin(region_peaks)][
                linger_col
            ].tolist()
        )

    # Calculate absolute cell cohort sizes for proper weighted averaging
    n8_g = (ds8_atac.obs["leiden_res_2.0"] == "2").sum()
    n9_g = (ds9_atac.obs["leiden_res_2.0"].isin(["6", "8"])).sum()
    n8_h = (ds8_atac.obs["leiden_res_2.0"].isin(["0", "1"])).sum()
    n9_h = (ds9_atac.obs["leiden_res_2.0"].isin(["2", "4", "5"])).sum()

    # Extract single-cell signals per dataset cohort
    sig8_g = np.asarray(
        ds8_atac[ds8_atac.obs["leiden_res_2.0"] == "2", region_peaks].X.mean(
            axis=0
        )
    ).flatten()
    sig9_g = np.asarray(
        ds9_atac[
            ds9_atac.obs["leiden_res_2.0"].isin(["6", "8"]), region_peaks
        ].X.mean(axis=0)
    ).flatten()
    sig8_h = np.asarray(
        ds8_atac[
            ds8_atac.obs["leiden_res_2.0"].isin(["0", "1"]), region_peaks
        ].X.mean(axis=0)
    ).flatten()
    sig9_h = np.asarray(
        ds9_atac[
            ds9_atac.obs["leiden_res_2.0"].isin(["2", "4", "5"]), region_peaks
        ].X.mean(axis=0)
    ).flatten()

    # Compute global cross-cohort weighted signal tracks
    sig_gpnmb = (sig8_g * n8_g + sig9_g * n9_g) / (n8_g + n9_g)
    sig_homeo = (sig8_h * n8_h + sig9_h * n9_h) / (n8_h + n9_h)

    # 3. Process Peak Differentials and Signatures
    res8 = result8[result8["names"].isin(region_peaks)].set_index("names")
    res9 = result9[result9["names"].isin(region_peaks)].set_index("names")

    def _get_peak_color(p):
        if p not in res8.index or p not in res9.index:
            return _C_NS
        r8, r9 = res8.loc[p], res9.loc[p]
        if (
            r8["pvals_adj"] < fdr_threshold
            and r9["pvals_adj"] < fdr_threshold
        ):
            return _C_GPNMB if r8["logfoldchanges"] > 0 else _C_HOMEO
        return _C_NS

    peak_colors = {p: _get_peak_color(p) for p in region_peaks}

    # 4. Axes Multi-Panel Canvas Setup
    fig, axes = plt.subplots(
        3, 1, figsize=figsize, sharex=True, gridspec_kw={"height_ratios": [2, 3, 3]}
    )
    ax_g, ax_gpnmb, ax_homeo = axes
    y_max = max(sig_gpnmb.max(), sig_homeo.max()) * 1.25
    ax_g.set_title(
        f"{chrom}:{start:,}—{end:,}",
        fontsize=FS_TITLE,
        fontweight="bold",
        pad=25,
    )

    # 5. Gene Annotation Track
    ax_g.set_xlim(0, width)
    ax_g.set_ylim(-0.5, max(len(gene_list), 1) * 0.7)
    for i, g_name in enumerate(gene_list):
        bits = region_gtf[region_gtf["gene_name"] == g_name]
        gene_row = bits[bits["feature"] == "gene"].iloc[0]
        strand, y = gene_row["strand"], i * 0.7
        c_exon, c_arrow = (
            (C_RED_EXON, C_RED_ARROW)
            if strand == "-"
            else (C_BLU_EXON, C_BLU_ARROW)
        )
        g_s, g_e = (
            max(0, gene_row["start"] - offset),
            min(width, gene_row["end"] - offset),
        )

        # Baseline gene locus spine
        ax_g.hlines(y, g_s, g_e, colors=c_arrow, lw=2.5, zorder=1)

        # Collapse exons using dynamic union algorithm
        exons = bits[bits["feature"] == "exon"][["start", "end"]].sort_values(
            "start"
        )
        if not exons.empty:
            merged_exons = []
            curr_s, curr_e = exons.iloc[0]["start"], exons.iloc[0]["end"]
            for _, row in exons.iloc[1:].iterrows():
                if row["start"] <= curr_e:
                    curr_e = max(curr_e, row["end"])
                else:
                    merged_exons.append((curr_s, curr_e))
                    curr_s, curr_e = row["start"], row["end"]
            merged_exons.append((curr_s, curr_e))

            # Render exon block geometry
            for s, e in merged_exons:
                es, ee = max(0, s - offset), min(width, e - offset)
                ax_g.add_patch(
                    Rectangle(
                        (es, y - 0.15),
                        ee - es,
                        0.3,
                        facecolor=c_exon,
                        edgecolor=c_exon,
                        linewidth=0.5,
                        zorder=2,
                    )
                )

        # Transcriptional direction indicator arrow
        arrow_x = g_e if strand == "+" else g_s
        ax_g.annotate(
            "",
            xy=(arrow_x, y),
            xytext=(
                arrow_x - (width * 0.03 if strand == "+" else -width * 0.03),
                y,
            ),
            arrowprops=dict(
                arrowstyle="->", color=c_arrow, lw=4, mutation_scale=25
            ),
            zorder=10,
        )
        ax_g.text(
            g_s,
            y + 0.22,
            g_name,
            fontsize=FS_GENES,
            fontweight="bold",
            fontstyle="italic",
            zorder=11,
        )
    ax_g.axis("off")

    # 6. Render Signal Tracks with Dual-Dataset Cohort Estimations
    def draw_track(ax, global_signals, sig_d8, sig_d9, is_gpnmb=False):
        ax.set_xlim(0, width)
        ax.set_ylim(0, y_max)

        for j, p_id in enumerate(region_peaks):
            ps = max(0, p_coords.loc[p_id, "s"] - offset)
            pe = min(width, p_coords.loc[p_id, "e"] - offset)
            midpoint = ps + (pe - ps) / 2

            # Main global peak envelope bar
            ax.add_patch(
                Rectangle(
                    (ps, 0),
                    pe - ps,
                    global_signals[j],
                    facecolor=peak_colors[p_id],
                    alpha=0.7 if peak_colors[p_id] != _C_NS else 0.35,
                )
            )

            # Overlay individual dataset empirical estimations if significant
            if peak_colors[p_id] != _C_NS:
                # Dataset 8 estimation marker: Black Circle
                ax.plot(
                    midpoint,
                    sig_d8[j],
                    marker="o",
                    color="black",
                    markersize=7,
                    alpha=0.8,
                    zorder=15,
                )
                # Dataset 9 estimation marker: Black Triangle
                ax.plot(
                    midpoint,
                    sig_d9[j],
                    marker="^",
                    color="black",
                    markersize=8,
                    alpha=0.8,
                    zorder=15,
                )

            # Hatch pattern for LINGER interactions
            if p_id in linger_cis_peaks:
                ax.add_patch(
                    Rectangle(
                        (ps, 0),
                        pe - ps,
                        global_signals[j],
                        facecolor="none",
                        edgecolor="#222222",
                        hatch="///",
                        alpha=0.3,
                        zorder=4,
                    )
                )

        lbl = "Activated\nClusters" if is_gpnmb else "Homeostatic\nClusters"
        ax.set_ylabel(lbl, fontsize=FS_LABELS, fontweight="bold", labelpad=15)
        ax.tick_params(labelsize=FS_SMALL)

    # Draw both tracks supplying global averages and raw dataset-specific arrays
    draw_track(ax_gpnmb, sig_gpnmb, sig8_g, sig9_g, is_gpnmb=True)
    draw_track(ax_homeo, sig_homeo, sig8_h, sig9_h)

    # ==============================================================================
    # 7. MANUSCRIPT COMPREHENSIVE LEGEND CONSTRUCTION
    # ==============================================================================
    handles = [
        mpatches.Patch(facecolor=_C_GPNMB, label='DAR Enriched in Activated'),
        mpatches.Patch(facecolor=_C_HOMEO, label='DAR Enriched in Homeostatic'),
        mpatches.Patch(facecolor=_C_NS, label='Non-Enriched Peak'),
        mpatches.Patch(facecolor='none', edgecolor='#222222', hatch='///', label='LINGER cis-element'),
        plt.Line2D([0], [0], marker='o', color='black', linestyle='', markersize=7, label='Dataset 8 Mean'),
        plt.Line2D([0], [0], marker='^', color='black', linestyle='', markersize=8, label='Dataset 9 Mean')
    ]
    
    # Render the consolidated legend outside the plot area to prevent overlap
    ax_homeo.legend(
        handles=handles, 
        fontsize=FS_SMALL, 
        loc='upper right', 
        frameon=True,
        bbox_to_anchor=(1.0, -0.25), 
        edgecolor='black'
    )

    # Configure genomic coordinate tick intervals across the horizontal span
    ax_homeo.set_xticks(np.linspace(0, width, 5))
    ax_homeo.set_xticklabels(
        [f"{int(t + offset):,}" for t in np.linspace(0, width, 5)], 
        fontsize=FS_SMALL
    )
    ax_homeo.tick_params(axis='x', pad=15)

    # Final aesthetic clean-up and rendering
    sns.despine(fig=fig, top=True, right=True)
    plt.tight_layout()
    plt.show()

In [ ]:
# NC -> CHR chromosome conversion
def convert_refseq_to_chr(chrom_str):
    chrom_str = str(chrom_str)
    # Mitochondial DNA
    if "NC_012920" in chrom_str:
        return "chrM"
    # Usual chromosomes
    if chrom_str.startswith("NC_0000"):
        try:
            num_str = chrom_str[7:9]
            num = int(num_str)
            if num == 23:
                return "chrX"
            elif num == 24:
                return "chrY"
            else:
                return f"chr{num}"
        except ValueError:
            return chrom_str
    return chrom_str

# Apply it to all dataframe
df_gtf["chrom"] = df_gtf["chrom"].apply(convert_refseq_to_chr)

In [ ]:
# S100 region
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom="chr1",  
    start=153510000,  
    end=153575000,  
    df_linger=df_linger_data,  
)

In [ ]:
# IQGAP2
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom="chr5",  
    start=76350000,  
    end=76750000,  
    df_linger=df_linger_data,  
)

In [ ]:
# ZNF804A
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom="chr2",  
    start=184590000,  
    end=184950000,  
    df_linger=df_linger_data,  
)

In [ ]:
# FOXP1
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom="chr3",  
    start=70900000,  
    end=71650000,  
    df_linger=df_linger_data,  
)

In [ ]:
# GPNMB
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom="chr7",  
    start=23225000,  
    end=23287500,  
    df_linger=df_linger_data,  
)

In [ ]:
# PTPRG
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr3', 
    start=61500000, 
    end=62250000,
    df_linger=df_linger_data,  
)

In [ ]:
# SLC11A1
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr2', 
    start=218375000, 
    end=218410000,
    df_linger=df_linger_data,  
)

In [ ]:
# SLC11A1
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr2', 
    start=218375000, 
    end=218410000,
    df_linger=df_linger_data,  
)

In [ ]:
# MYO1E
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr15', 
    start=59100000, 
    end=59400000,
    df_linger=df_linger_data,  
)

In [ ]:
# STARD13
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr13', 
    start=33000000, 
    end=33800000,
    df_linger=df_linger_data,  
)

In [ ]:
# DPYD
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr1', 
    start=97000000, 
    end=98000000,
    df_linger=df_linger_data,  
)

In [ ]:
# KCNMA1
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr10', 
    start=76800000, 
    end=77700000,
    df_linger=df_linger_data,  
)

In [ ]:
# ATG7
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr3', 
    start=11250000, 
    end=11600000,
    df_linger=df_linger_data,  
)

In [ ]:
# MEIS1
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom="chr2",  
    start=66400000,  
    end=66600000,  
    df_linger=df_linger_data,  
)

Thus, while signature genes and the S100A2-6 regions are more open in GPNMB+ clusters, MEIS1 exhibits higher accessibility in homeostatic nuclei.

# **Analyzing the expression of S100As**

We decided to additionally evaluate whether genes within the S100A locus also exhibit higher expression in GPNMB+ cells than in homeostatic ones. To achieve this, we applied a paired pseudobulk analysis, directly comparing GPNMB+ and homeostatic cells within each individual donor.

In [ ]:
# Run pseudobulk for S100A2, S100A3, S100A4, S100A5, S100A6

def run_rna_pseudobulk(
    adata_obj, 
    dataset_name, 
    cluster_col, 
    act_clusters, 
    homeo_clusters, 
    donor_col, 
    target_genes
):
    """
    Computes pseudobulk expression values profiles per donor/sample 
    and evaluates statistical significance using a paired t-test.
    """
    print(f"Processing RNA Pseudobulk for {dataset_name}...")
    
    # 1. Filter cells to retain only target functional microglial states
    all_target_clusters = act_clusters + homeo_clusters
    cell_mask = adata_obj.obs[cluster_col].isin(all_target_clusters)
    adata_filtered = adata_obj[cell_mask].copy()
    
    # Assign definitive functional state labels
    adata_filtered.obs['functional_state'] = adata_filtered.obs[cluster_col].apply(
        lambda x: 'Activated' if x in act_clusters else 'Homeostasis'
    )
    
    # 2. Match target gene symbols with matrix feature index (Ensembl IDs or Symbols)
    symbol_to_var = dict(zip(adata_filtered.var['gene_symbols'], adata_filtered.var_names))
    valid_genes = [g for g in target_genes if g in symbol_to_var]
    
    if not valid_genes:
        print(f"Warning: None of the target genes found in 'gene_symbols' for {dataset_name}!")
        return pd.DataFrame()
        
    # Map symbols back to internal matrix feature identifiers
    feature_ids = [symbol_to_var[g] for g in valid_genes]
    
    # 3. Extract single-cell expression matrices safely handling sparse inputs
    if hasattr(adata_filtered.X, "toarray"):
        exp_matrix = adata_filtered[:, feature_ids].X.toarray()
    else:
        exp_matrix = np.asarray(adata_filtered[:, feature_ids].X)
        
    # Build core tracking dataframe mapping human-readable symbols to columns
    df_cells = pd.DataFrame(exp_matrix, columns=valid_genes, index=adata_filtered.obs_names)
    df_cells['donor_id'] = adata_filtered.obs[donor_col].astype(str)
    df_cells['state'] = adata_filtered.obs['functional_state']
    
    # 4. Collapse profiles into donor-level means within each state
    df_pseudobulk = df_cells.groupby(['donor_id', 'state'])[valid_genes].mean().reset_index()
    
    # Retain strictly paired biological donors across both microglial states
    donor_counts = df_pseudobulk.groupby('donor_id')['state'].nunique()
    paired_donors = donor_counts[donor_counts == 2].index.tolist()
    df_paired = df_pseudobulk[df_pseudobulk['donor_id'].isin(paired_donors)]
    
    print(f"-> Found {len(paired_donors)} independent biological donors with paired profiles.")
    if df_paired.empty:
        return pd.DataFrame()
        
    # 5. Statistical evaluation via Paired Student's t-test
    results = []
    for gene in valid_genes:
        act_values = df_paired[df_paired['state'] == 'Activated'].sort_values('donor_id')[gene].values
        homeo_values = df_paired[df_paired['state'] == 'Homeostasis'].sort_values('donor_id')[gene].values
        
        mean_act = np.mean(act_values)
        mean_homeo = np.mean(homeo_values)
        
        # Handle mathematical edge cases for division by zero smoothly
        if mean_act == 0 and mean_homeo == 0:
            lfc, p_val = 0.0, 1.0
        else:
            lfc = np.log2((mean_act + 1e-6) / (mean_homeo + 1e-6))
            _, p_val = ttest_rel(act_values, homeo_values)
            
        results.append({
            'Dataset': dataset_name,
            'Gene': gene,
            'Mean_Homeostasis': round(mean_homeo, 4),
            'Mean_Activated': round(mean_act, 4),
            'Log2FC': round(lfc, 3),
            'p_value': p_val
        })
        
    return pd.DataFrame(results)

# ==============================================================================
# EXECUTION PIPELINE FOR THE MULTI-DATASET RNA ANALYSIS
# ==============================================================================
target_s100_genes = ['S100A2', 'S100A3', 'S100A4', 'S100A5', 'S100A6']

# Run pseudobulk statistics on pure RNA object for Dataset 8
res_d8_rna = run_rna_pseudobulk(
    adata_obj=ds8,
    dataset_name="Dataset 8",
    cluster_col="leiden_res_2.0",
    act_clusters=["2"],
    homeo_clusters=["0", "1"],
    donor_col="sample",
    target_genes=target_s100_genes
)

# Run pseudobulk statistics on pure RNA object for Dataset 9
res_d9_rna = run_rna_pseudobulk(
    adata_obj=ds9,
    dataset_name="Dataset 9",
    cluster_col="leiden_res_2.0",
    act_clusters=["6", "8"],
    homeo_clusters=["2", "4", "5"],
    donor_col="sample",  # Unified across datasets to match RNA metadata
    target_genes=target_s100_genes
)

# Consolidate individual results and apply global FDR Benjamini-Hochberg correction
valid_dfs = [df for df in [res_d8_rna, res_d9_rna] if df is not None and not df.empty]
if valid_dfs:
    df_rna_final = pd.concat(valid_dfs, ignore_index=True)
    df_rna_final['p_value'] = df_rna_final['p_value'].fillna(1.0)
    _, p_adjs, _, _ = multipletests(df_rna_final['p_value'], method='fdr_bh')
    df_rna_final['p_adj'] = p_adjs
else:
    df_rna_final = pd.DataFrame()
    print("Error: Both processed DataFrames returned empty. Double-check inputs.")

# Display final formatted dataframe structure
df_rna_final

In [ ]:
# Plot the expression of S100A4 in GPNMB+ vs homeostatic cells

def plot_gene_expression(adata_8, adata_9, target_gene):
    """
    Generates a high-resolution violin plot overlaid with single-cell expression
    dots for the S100A4 gene across Dataset 8 and Dataset 9 microglial states.
    """
    # 1. Define manuscript-standard publication color palettes
    violin_palette = {"Activated": "#f39c12", "Homeostatic": "#9b59b6"}
    dots_palette = {"Activated": "#FF7F00", "Homeostatic": "#A020F0"}
    
    # Define multi-dataset experimental setup
    experimental_setup = [
        ("D8", adata_8, ["2"], ["0", "1"]),
        ("D9", adata_9, ["6", "8"], ["2", "4", "5"]),
    ]
    
    extracted_data = []
    
    # 2. Extract single-cell vector expressions from pure RNA matrices
    for dataset_name, adata_obj, active_clusters, homeostatic_clusters in experimental_setup:
        # Map human-readable symbols to internal matrix indices (Ensembl IDs)
        symbol_to_var = dict(zip(adata_obj.var["gene_symbols"], adata_obj.var_names))
        
        if target_gene not in symbol_to_var:
            continue
            
        feature_id = symbol_to_var[target_gene]
        
        # Extract and flatten the expression vector handling sparse formats safely
        if hasattr(adata_obj.X, "toarray"):
            expression_vector = adata_obj[:, feature_id].X.toarray().flatten()
        else:
            expression_vector = np.asarray(adata_obj[:, feature_id].X).flatten()
            
        # Retain only target functional microglial states
        target_clusters = active_clusters + homeostatic_clusters
        cell_mask = adata_obj.obs["leiden_res_2.0"].isin(target_clusters)
        
        # Construct localized slice dataframe for the target gene
        df_slice = pd.DataFrame({
            "Expression": expression_vector[cell_mask],
            "Cluster": adata_obj.obs.loc[cell_mask, "leiden_res_2.0"].astype(str),
            "Dataset": dataset_name
        })
        
        # Partition individual cell clusters into clear functional phenotypes
        df_slice["Group"] = df_slice["Cluster"].apply(
            lambda x: "Activated" if x in active_clusters else "Homeostatic"
        )
        extracted_data.append(df_slice)
        
    if not extracted_data:
        print(f"Error: {target_gene} could not be resolved from dataset matrices.")
        return
        
    # Consolidate individual data structures into a unified long-form dataframe
    df_plot = pd.concat(extracted_data, ignore_index=True)
    
    # 3. Multi-panel figures canvas initialization
    plt.figure(figsize=(8, 4.5), dpi=300)
    
    # Render desaturated probability density envelopes as background anchors
    sns.violinplot(
        data=df_plot,
        x="Dataset",
        y="Expression",
        hue="Group",
        split=False,
        inner=None,
        palette=violin_palette,
        linewidth=1.2,
        alpha=0.5,
        width=0.65,
        cut=0
    )
    
    # Overlay ultra-bright single-cell expression dots with custom contours
    sns.stripplot(
        data=df_plot,
        x="Dataset",
        y="Expression",
        hue="Group",
        palette=dots_palette,
        dodge=True,
        size=2.5,
        alpha=0.65,
        jitter=0.25,
        edgecolor="#111111",
        linewidth=0.4,
        zorder=3
    )
    
    # 4. Typography, formatting and aesthetic layouts polish
    plt.title(f"Single-Cell Expression Profile of {target_gene}", fontsize=14, fontweight="bold", pad=15)
    plt.ylabel("Normalized Expression Level", fontsize=12)
    plt.xlabel("")
    
    # Establish dynamic defensive axis buffers to cleanly encapsulate dropouts
    plt.ylim(-0.15, df_plot["Expression"].max() * 1.08)
    
    # Re-inject unified clean figure legend avoiding duplicate keys
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.legend(
        handles[:2], 
        labels[:2], 
        loc="upper right", 
        frameon=True, 
        facecolor="white", 
        edgecolor="none"
    )
    
    sns.despine()
    plt.tight_layout()
    plt.show()
    
# Execute plot function generation for the manuscript
plot_gene_expression(ds8, ds9, 'S100A4')

Thus, out of all S100A genes, S100A4 exhibits the highest log2 fold change in GPNMB+ cells compared to homeostatic ones. While this trend is nominally significant in both individual datasets (p-value = 0.0127 for Dataset 8 and 0.0294 for Dataset 9), it does not retain strict statistical significance after global FDR correction across the locus.